# Collaborative filtering
Student information here!!!

In [22]:
from IPython.core.display import HTML
from movie_display import movie_display
import pandas as pd
import numpy as np
from math import sqrt

## Dataset analysis

In [23]:
# Your code
df = pd.read_csv("./dataset/ratings.csv")
user_item = df.pivot(index="userId", columns="movieId", values="rating")
print(df.head(5))

def cosineSimilarity(object1,object2):
        object1 = np.nan_to_num(object1)
        object2 = np.nan_to_num(object2)
        # sum_object1 = 0
        # sum_object2 = 0
        # for i in range(len(object1)):
        #     sum_object1 += object1[i]**2
        #     sum_object2 += object2[i]**2

        # denominator = sqrt(sum_object1) * sqrt(sum_object2)
        denominator = np.linalg.norm(object1) * np.linalg.norm(object2)
        cosSim= object1 @ object2 / denominator if denominator else 0.0
        return cosSim

def pearsonSim(a,b):
    # only use positions where BOTH rated, centred on their own mean
    # both = ~np.isnan(a) & ~np.isnan(b)
    
    # if both.sum() < 2:
    #     return 0.0
    
    a = np.nan_to_num(a - np.nanmean(a))
    b = np.nan_to_num(b - np.nanmean(b))
    
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return a @ b / denom if denom else 0.0

def adjustedCosSim(a, b, item_means):
    a = np.nan_to_num(a - item_means)
    b = np.nan_to_num(b - item_means)

    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return a @ b / denom if denom else 0.0

# slide example: users A-D, movies HP1 HP2 HP3 TW SW1 SW2 SW3
nan = np.nan
slide = np.array([[4, nan, nan, 5,   1,   nan, nan],
                  [5, 5,   4,   nan, nan, nan, nan],
                  [nan, nan, nan, 2,   4,   5,   nan],
                  [nan, 3,   nan, nan, nan, nan, 3]])

print("Cosine  A,B:", cosineSimilarity(slide[0], slide[1]))   # slides: 0.38
print("Pearson A,B:", pearsonSim(slide[0], slide[1]))         # slides: 0.092
item_means = np.nanmean(slide, axis=0)                        # one mean per movie
print("Adj cos A,B:", adjustedCosSim(slide[0], slide[1], item_means))  # slides: -0.1025

def similarity_matrix(M, metric="cosine"):
    """Similarity between every pair of ROWS of M (NaN = not rated)."""
    M = M.astype(np.float32)
   
    if metric == "pearson":                    # subtract each vector's own mean
        M = M - np.nanmean(M, axis=1, keepdims=True)
    elif metric == "adjusted cosine":          # subtract the mean of the other dimension
        M = M - np.nanmean(M, axis=0, keepdims=True)
    
    X = np.nan_to_num(M)                       # missing -> 0
    
    norms = np.linalg.norm(X, axis=1)
    norms[norms == 0] = 1                      # zero vector -> similarity 0
    
    X = X / norms[:, None]
    S = X @ X.T                                # all pairs at once
    np.fill_diagonal(S, 1)
    
    return S

R = user_item.to_numpy(dtype=np.float32)       # rows = users, columns = movies
user_sims = {m: similarity_matrix(R, m) for m in ["cosine", "pearson", "adjusted cosine"]}  # 671x671
item_sims = {}                                  # filled on demand: each is 9066x9066, ~330 MB

def get_item_sims(metric):
    if metric not in item_sims:
        item_sims[metric] = similarity_matrix(R.T, metric)
    return item_sims[metric]

   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205
Cosine  A,B: 0.3798685881987931
Pearson A,B: 0.09245003270420475
Adj cos A,B: -0.1025978352085154


## User-user recommendations

In [ ]:
# Your code
def recommend_user_user(R, S_user, u, n=10, k=10):
    sims = S_user[u].copy(); 
    sims[u] = -np.inf
    
    nb = np.argsort(-sims)[:k]                 # the k most similar users
    nb = nb[sims[nb] > 0]                      # only positively similar ones
    
    w = sims[nb]
    r = R[nb]
    rated = ~np.isnan(r)
    num = (np.nan_to_num(r) * w[:, None]).sum(axis=0)
    den = (rated * w[:, None]).sum(axis=0)
    count = rated.sum(axis=0)                  # how many neighbours rated each movie
    ok = np.isnan(R[u]) & (den > 0)            # unseen by u, rated by >= 1 neighbour
    pred = np.full(R.shape[1], np.nan); pred[ok] = num[ok] / den[ok]
    cand = np.where(ok)[0]
    order = cand[np.lexsort((-count[cand], -pred[cand]))][:n]
    
    return order, pred[order]

## Item-item recommendations

In [ ]:
# Your code
def recommend_item_item(R, S_item, u, n=10, min_rating=3.5, j=10):
    liked = np.where(R[u] >= min_rating)[0]    # movies u rated >= 3.5
    
    if len(liked) == 0:
        return np.array([], int), np.array([])
    
    sub = S_item[:, liked].copy()              # every movie vs the liked ones
    sub[sub < 0] = 0
    
    if len(liked) > j:                         # keep the j most similar liked movies per candidate
        thresh = -np.partition(-sub, j - 1, axis=1)[:, j - 1:j]
        sub[sub < thresh] = 0
    
    num, den = sub @ R[u, liked], sub.sum(axis=1)
    ok = np.isnan(R[u]) & (den > 0)
    pred = np.full(R.shape[1], np.nan); pred[ok] = num[ok] / den[ok]
    cand = np.where(ok)[0]
    order = cand[np.lexsort((-den[cand], -pred[cand]))][:n]
    
    return order, pred[order]